# Session 3: traverse computation, adjustment and error propagation
**A. Real-world problem.** To fix the corners of a plot, the surveyor walks round it:
measure the angle at each corner and the distance of each side (a *closed traverse*).
Because every measurement has small errors, the traverse does not quite return to its start.
The gap is the *misclosure*. We must (1) measure it, (2) judge if it is acceptably small,
(3) spread it out sensibly (the *Bowditch* or compass rule), (4) compute the area, and
(5) discover when a *blunder* (a wrong reading) is hiding behind a tidy adjustment.

**Plan.** Build a known pentagon, simulate field measurements with typical instrument errors,
adjust, and check every step against the known exact answer.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# A plot with exactly known coordinates (easting, northing in metres), walked clockwise:
# a 200 m x 120 m rectangle with a gable on top. Exact area = 200*120 + 0.5*200*60 = 30000 m^2.
TRUTH = np.array([[0.0, 0.0], [0.0, 120.0], [100.0, 180.0], [200.0, 120.0], [200.0, 0.0]])
TRUE_AREA = 30000.0
# Typical figures for a mid-range total station (they vary by instrument; check a datasheet):
ANGLE_SIGMA_ARCSEC = 20.0   # standard deviation of one measured angle
DIST_MM, DIST_PPM = 5.0, 5.0   # distance standard deviation = 5 mm + 5 ppm of the distance

## Step 1: legs, bearings and interior angles of the known polygon
Leg i runs from corner i to corner i+1 (the last leg returns to corner 1). Bearings are
clockwise from north. Walking clockwise, the next bearing is: previous bearing + 180 - interior
angle at the corner. Check by hand at corner 2: the legs are north (0 degrees) then bearing
59.04 degrees, so the interior angle is 0 + 180 - 59.04 = 120.96 degrees.

In [ ]:
def legs(pts):
    """Bearings (degrees) and lengths (m) of the closed polygon pts."""
    d = np.roll(pts, -1, axis=0) - pts
    return np.degrees(np.arctan2(d[:, 0], d[:, 1])) % 360.0, np.hypot(d[:, 0], d[:, 1])


def interior_angles(bearings):
    """Interior angle at each corner (degrees) for a clockwise polygon: angle i is between leg i-1 and leg i."""
    return (np.roll(bearings, 1) + 180.0 - bearings) % 360.0


def bearings_from_angles(b0, angles):
    """Bearing of every leg, given the bearing b0 of leg 1 and the interior angles at corners 2..n."""
    b = [b0 % 360.0]
    for k in range(1, len(angles)):
        b.append((b[-1] + 180.0 - angles[k]) % 360.0)
    return np.array(b)


def walk(start, bearings, dists):
    """Coordinates of the n+1 points visited; the last one should be the start again."""
    r = np.radians(bearings)
    steps = np.stack([dists * np.sin(r), dists * np.cos(r)], axis=1)   # departures, latitudes
    return np.vstack([start, start + np.cumsum(steps, axis=0)])

## Step 2: misclosure, relative accuracy, and the angle check
- Linear misclosure vector = (last point) - (start). Its length is the misclosure.
- Relative accuracy = perimeter / misclosure, quoted as "1 in N". A common quality target
  for ordinary work is 1 in 5000.
- Angle check: interior angles of an n-sided polygon must sum to (n - 2) x 180 degrees.

In [ ]:
def misclosure(pts):
    v = pts[-1] - pts[0]
    return v, float(np.hypot(*v))


def relative_accuracy(mis, perimeter):
    """The N in '1 in N'. Infinite for a perfect closure."""
    return np.inf if mis == 0 else perimeter / mis


def angle_misclosure(angles):
    """Sum of measured interior angles minus (n - 2) x 180, in degrees."""
    return float(np.sum(angles) - (len(angles) - 2) * 180.0)


def spread_angle_error(angles):
    """Share the angular misclosure equally between all angles."""
    return np.asarray(angles) - angle_misclosure(angles) / len(angles)

## Step 3: the Bowditch (compass) rule
Leg i has departure dE_i = D_i sin(b_i) and latitude dN_i = D_i cos(b_i). If the total
misclosure is (eE, eN) and the perimeter is P, leg i gets the correction

    cE_i = - eE D_i / P        cN_i = - eN D_i / P

so the corrections are proportional to leg length and add up to exactly -e. Therefore the
adjusted traverse closes exactly: sum(dE_i + cE_i) = eE - eE = 0.

In [ ]:
def bowditch(start, bearings, dists):
    """Adjusted points (n+1 of them) and the misclosure vector that was removed."""
    r = np.radians(bearings)
    steps = np.stack([dists * np.sin(r), dists * np.cos(r)], axis=1)
    e = steps.sum(axis=0)
    corr = -np.outer(dists / dists.sum(), e)
    return np.vstack([start, start + np.cumsum(steps + corr, axis=0)]), e

## Step 4: area by the coordinate (shoelace) method
Area = half the absolute value of sum over corners of (E_i N_{i+1} - E_{i+1} N_i).

In [ ]:
def shoelace(pts):
    p = np.asarray(pts, dtype=float)
    if np.allclose(p[0], p[-1]):
        p = p[:-1]
    e, n = p[:, 0], p[:, 1]
    return 0.5 * abs(float(np.dot(e, np.roll(n, -1)) - np.dot(n, np.roll(e, -1))))


def adjust(start, b0, angles, dists):
    """Full pipeline: spread the angle error, walk, Bowditch, area. Returns a dict."""
    angles = np.asarray(angles, dtype=float)
    dists = np.asarray(dists, dtype=float)
    ang_mis = angle_misclosure(angles)
    adj = spread_angle_error(angles)
    raw = walk(start, bearings_from_angles(b0, adj), dists)
    vec, mis = misclosure(raw)
    pts, _ = bowditch(start, bearings_from_angles(b0, adj), dists)
    return {"angle_misclosure_deg": ang_mis, "misclosure_vec": vec, "misclosure": mis,
            "relative": relative_accuracy(mis, float(dists.sum())), "raw": raw, "points": pts,
            "area": shoelace(pts), "perimeter": float(dists.sum())}

## Step 5: simulate field work with typical instrument errors
Measured = true + random error. Angle error has standard deviation 20 arc-seconds (1 arc-second
= 1/3600 degree); the distance error is 5 mm + 5 ppm of the distance. Seeded for repeatability.

In [ ]:
def distance_sigma(d):
    """Standard deviation (m) of a measured distance d (m): 5 mm + 5 ppm."""
    return DIST_MM * 1e-3 + DIST_PPM * 1e-6 * np.asarray(d, dtype=float)


def simulate_field(pts, rng, angle_arcsec=ANGLE_SIGMA_ARCSEC, dist_scale=1.0):
    b, d = legs(pts)
    a = interior_angles(b)
    ma = a + rng.normal(0.0, angle_arcsec / 3600.0, size=a.shape)
    md = d + rng.normal(0.0, 1.0, size=d.shape) * distance_sigma(d) * dist_scale
    return b[0], ma, md


def monte_carlo(pts=TRUTH, trials=4000, seed=2024, angle_arcsec=ANGLE_SIGMA_ARCSEC, dist_scale=1.0):
    """Repeat the survey many times. Returns arrays: raw misclosure length, relative N,
    area error, and error of the adjusted coordinates of the last corner."""
    rng = np.random.default_rng(seed)
    mis, rel, aerr, perr, uns = [], [], [], [], []
    for _ in range(trials):
        b0, ma, md = simulate_field(pts, rng, angle_arcsec, dist_scale)
        res = adjust(pts[0], b0, ma, md)
        mis.append(res["misclosure"])
        rel.append(res["relative"])
        aerr.append(res["area"] - shoelace(pts))
        perr.append(np.hypot(*(res["points"][-2] - pts[-1])))
        uns.append(misclosure(walk(pts[0], bearings_from_angles(b0, ma), md))[1])
    return {"misclosure": np.array(mis), "relative": np.array(rel), "area_err": np.array(aerr),
            "corner_err": np.array(perr), "raw_unadjusted": np.array(uns)}


def predicted_misclosure_rms(pts=TRUTH, angle_arcsec=ANGLE_SIGMA_ARCSEC, dist_scale=1.0):
    """First-order (small error) prediction of the RMS misclosure when the angles are NOT adjusted.
    Uses the Jacobian of the walk with respect to the n-1 angles and n distances."""
    b, d = legs(pts)
    a = interior_angles(b)
    n = len(d)

    def f(x):
        ang = np.concatenate([[a[0]], x[:n - 1]])
        return misclosure(walk(pts[0], bearings_from_angles(b[0], ang), x[n - 1:]))[0]

    x0 = np.concatenate([a[1:], d])
    h = 1e-6
    J = np.stack([(f(x0 + h * np.eye(len(x0))[i]) - f(x0 - h * np.eye(len(x0))[i])) / (2 * h)
                  for i in range(len(x0))], axis=1)
    var = np.concatenate([np.full(n - 1, (angle_arcsec / 3600.0) ** 2), (distance_sigma(d) * dist_scale) ** 2])
    cov = J @ np.diag(var) @ J.T
    return float(np.sqrt(np.trace(cov)))


def max_angle_sigma(rms_limit, pts=TRUTH):
    """Design helper: the largest angle standard deviation (arc-seconds) whose predicted RMS misclosure
    (metres) stays at or below rms_limit, found by bisection."""
    lo, hi = 0.0, 1000.0
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if predicted_misclosure_rms(pts, angle_arcsec=mid) > rms_limit:
            hi = mid
        else:
            lo = mid
    return lo

## Step 6: curvature and refraction for long sights
The Earth curves away from a horizontal line of sight, and the air bends the ray slightly back.
Over K kilometres, the combined correction is c = 0.0675 K^2 metres. Where it comes from:
the curvature part is d^2 / (2R) (R = 6371 km); refraction removes a fraction k, about 0.14,
of it, leaving (1 - k) K^2 / (2R). Refraction varies with the weather; 0.14 is a typical value.

In [ ]:
R_EARTH = 6371000.0   # metres, mean radius of the Earth
K_REFRACTION = 0.14   # typical coefficient of refraction (varies)


def sphere_drop(d):
    """Exact height (m) of a horizontal tangent line above the sphere at distance d (m) along the tangent:
    sqrt(R^2 + d^2) - R, written to avoid cancellation."""
    d = np.asarray(d, dtype=float)
    return d * d / (np.sqrt(R_EARTH ** 2 + d * d) + R_EARTH)


def cr_formula(k_km):
    """The standard small-correction formula, metres."""
    return 0.0675 * np.asarray(k_km, dtype=float) ** 2


def cr_physics(k_km, k=K_REFRACTION):
    """Curvature from the sphere geometry, reduced by refraction, metres."""
    return (1.0 - k) * sphere_drop(np.asarray(k_km, dtype=float) * 1000.0)

## Step 7: a blunder that the adjustment hides, and how to find it
A blunder is a wrong reading (not a small random error). Suppose the angle at one corner is
read 1 degree too large. The traverse then fails to close by a large amount; Bowditch happily
spreads it and the adjusted figure closes *exactly*, but the corners are wrong. To find the
corner: an angle blunder at corner k turns every later leg about corner k. So the misclosure
vector is perpendicular to the line from corner k to the start. The corner whose line to the
start is most nearly perpendicular to the misclosure is the suspect.

In [ ]:
def locate_angle_blunder(start, b0, angles, dists):
    """Return (index of suspect corner, estimated size of the blunder in degrees) for corners 2..n,
    using the unadjusted angles."""
    raw = walk(start, bearings_from_angles(b0, angles), dists)
    mis = raw[-1] - raw[0]
    best, score = None, 2.0
    for k in range(1, len(angles)):
        v = raw[0] - raw[k]
        c = abs(np.dot(mis, v)) / (np.hypot(*mis) * np.hypot(*v))
        if c < score:
            best, score = k, c
    v = raw[0] - raw[best]
    size = 2.0 * np.degrees(np.arcsin(min(1.0, np.hypot(*mis) / (2.0 * np.hypot(*v)))))
    return best, float(size)


def with_blunder(angles, corner, size_deg):
    a = np.array(angles, dtype=float)
    a[corner] += size_deg
    return a

## Step 8: the field data for the mini-project (synthetic)
A six-corner site walked clockwise from corner 1 at E 5000.000, N 10000.000 m. The bearing of
leg 1 is known. `FIELD_ANGLES` are the six measured interior angles (degrees), `FIELD_DISTS`
the six leg lengths (m). One angle was misread. `FIELD_TRUTH` is the answer key (do not peek
until you have your own coordinates).

In [ ]:
FIELD_START = np.array([5000.0, 10000.0])
FIELD_B0 = 351.8699
FIELD_ANGLES = [109.4402, 123.8647, 131.9041, 114.8573, 119.7432, 121.1976]
FIELD_DISTS = [141.425, 134.536, 140.806, 131.234, 162.797, 152.97]
FIELD_TRUTH = np.array([[5000.0, 10000.0], [4980.0, 10140.0], [5080.0, 10230.0],
                        [5220.0, 10215.0], [5260.0, 10090.0], [5150.0, 9970.0]])

## Step 9: see it
Left: the walked traverse with its misclosure (exaggerated errors, so you can see them).
Right: the Bowditch corrections move each corner back along the walk.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    b, d = legs(TRUTH)
    print("bearings", b.round(2), "lengths", d.round(3), "perimeter", round(float(d.sum()), 3))
    print("interior angles", interior_angles(b).round(2), "sum", round(float(interior_angles(b).sum()), 6))

    rng = np.random.default_rng(5)
    b0, ma, md = simulate_field(TRUTH, rng, angle_arcsec=20.0 * 60, dist_scale=40.0)   # errors exaggerated for the picture
    res = adjust(TRUTH[0], b0, ma, md)
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    t = np.vstack([TRUTH, TRUTH[0]])
    ax.plot(t[:, 0], t[:, 1], "k:", label="true corners")
    ax.plot(res["raw"][:, 0], res["raw"][:, 1], "o-", color="tab:red", label="walked (unadjusted)")
    ax.plot(res["points"][:, 0], res["points"][:, 1], "s-", color="tab:green", label="after Bowditch")
    ax.annotate("", xy=res["raw"][-1], xytext=res["raw"][0], arrowprops=dict(arrowstyle="->", color="tab:red"))
    ax.set_aspect("equal")
    ax.set_xlabel("easting (m)")
    ax.set_ylabel("northing (m)")
    ax.set_title("Misclosure (red arrow) removed by Bowditch (errors exaggerated)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(OUT / "bowditch.png", dpi=120)

    mc = monte_carlo()
    print(f"typical errors: RMS raw misclosure {np.sqrt(np.mean(mc['raw_unadjusted'] ** 2)) * 1000:.1f} mm; "
          f"predicted {predicted_misclosure_rms() * 1000:.1f} mm; "
          f"median relative accuracy 1 in {np.median(mc['relative']):.0f}; "
          f"area error std {mc['area_err'].std():.3f} m2")

    sig = np.array([5.0, 10.0, 20.0, 40.0, 80.0, 160.0])
    rms = [np.sqrt(np.mean(monte_carlo(trials=1500, angle_arcsec=s)["raw_unadjusted"] ** 2)) * 1000 for s in sig]
    fig2, ax2 = plt.subplots(figsize=(6.2, 4))
    ax2.loglog(sig, rms, "o-", label="Monte Carlo RMS misclosure")
    ax2.axhline(673.238 / 5000 * 1000, color="red", ls="--", label="1 in 5000 limit (134.6 mm)")
    ax2.set_xlabel("angle standard deviation (arc-seconds)")
    ax2.set_ylabel("RMS misclosure (mm)")
    ax2.set_title("Sensitivity: angle error drives the misclosure")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "sensitivity.png", dpi=120)

    K = np.linspace(0, 3, 61)
    fig3, ax3 = plt.subplots(figsize=(6.2, 4))
    ax3.plot(K, sphere_drop(K * 1000) * 1000, label="curvature only (sphere geometry)")
    ax3.plot(K, cr_formula(K) * 1000, "--", label="0.0675 K^2 (curvature and refraction)")
    ax3.set_xlabel("sight length K (km)")
    ax3.set_ylabel("correction (mm)")
    ax3.set_title("Long sights: the Earth curves away")
    ax3.grid(alpha=0.3)
    ax3.legend(fontsize=8)
    fig3.tight_layout()
    fig3.savefig(OUT / "curvature.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), do the mini-project
in the README, then run the checkpoint.

## Coding tasks
1. **Easy.** Run `monte_carlo(angle_arcsec=10.0)` and `monte_carlo(angle_arcsec=40.0)` and compare the median relative accuracy. Predict the order first.
2. **Stretch.** Write `corner_errors(result, truth)` that returns the distance of each adjusted corner from its true position, and use it on the blunder case of Section 7.
3. **Explain.** Explain why the locator uses the line from the corner to the start, and why the misclosure vector is perpendicular to it for an angle blunder.

## Assessment
- **A. Conceptual.** The Bowditch rule always makes a traverse close exactly. Does a perfect closure after adjustment prove that the survey is correct? Give an example from this session.
- **B. Numerical.** A traverse of 673.238 m perimeter closes with a misclosure of 0.050 m. What is the relative accuracy? How much is a leg of 116.619 m corrected?
- **C. Graph and simulation interpretation.** Read the sensitivity graph: roughly how large can the angle error become before the typical misclosure exceeds the 1 in 5000 limit? What does the straight rise tell you about the effect of a better instrument?
- **D. Design and optimisation challenge.** Design: choose the poorest angle standard deviation (in arc-seconds) so that about 95 percent of surveys of this plot meet 1 in 5000. Hint: require the RMS misclosure to be at most half the limit. Why half, and what is the trade-off?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    T = L.TRUTH
    b, d = L.legs(T)
    a = L.interior_angles(b)
    P = float(d.sum())
    checks.append(("worked example: bearings 0, 59.04, 120.96, 180, 270 degrees",
                   np.allclose(b, [0.0, 59.04, 120.96, 180.0, 270.0], atol=5e-3)))
    checks.append(("worked example: leg lengths 120, 116.619, 116.619, 120, 200 m, perimeter 673.238 m",
                   np.allclose(d, [120.0, 116.619, 116.619, 120.0, 200.0], atol=5e-4) and abs(P - 673.238) < 5e-4))
    checks.append(("interior angles 90, 120.96, 118.07, 120.96, 90 degrees sum to 540 = (5 - 2) x 180",
                   np.allclose(a, [90.0, 120.96, 118.07, 120.96, 90.0], atol=5e-3) and abs(a.sum() - 540.0) < 1e-9))
    checks.append(("exact shoelace area of the plot is 30000 m2 (rectangle 24000 + gable 6000)",
                   abs(L.shoelace(T) - 30000.0) < 1e-9 and L.TRUE_AREA == 30000.0))
    checks.append(("shoelace gives 100 for a 10 m square, closed or open list",
                   abs(L.shoelace([[0, 0], [0, 10], [10, 10], [10, 0]]) - 100.0) < 1e-12
                   and abs(L.shoelace([[0, 0], [0, 10], [10, 10], [10, 0], [0, 0]]) - 100.0) < 1e-12))
    pts = L.walk(T[0], L.bearings_from_angles(b[0], a), d)
    checks.append(("validation: error-free angles and distances close to better than 1e-9 m",
                   L.misclosure(pts)[1] < 1e-9 and np.allclose(pts[:-1], T, atol=1e-9)))

    # hand calculation of the Bowditch correction
    e = np.array([0.100, -0.060])
    corr_leg5 = -e * 200.0 / P
    checks.append(("hand calculation: misclosure (0.100, -0.060) m, leg 5 of 200 m gets (-0.0297, +0.0178) m",
                   np.allclose(corr_leg5, [-0.02971, 0.01783], atol=5e-5)))
    rng = np.random.default_rng(3)
    b0, ma, md = L.simulate_field(T, rng, angle_arcsec=20.0 * 60, dist_scale=40.0)
    bearings = L.bearings_from_angles(b0, ma)
    raw = L.walk(T[0], bearings, md)
    adj, e2 = L.bowditch(T[0], bearings, md)
    checks.append(("Bowditch: the misclosure was not zero before, and the adjusted traverse closes exactly (< 1e-9 m)",
                   L.misclosure(raw)[1] > 0.5 and L.misclosure(adj)[1] < 1e-9))
    steps_before = np.diff(raw, axis=0)
    steps_after = np.diff(adj, axis=0)
    corr = steps_after - steps_before
    checks.append(("Bowditch: corrections are proportional to leg length and add up to minus the misclosure",
                   np.allclose(corr / md[:, None], corr[0] / md[0], atol=1e-12) and np.allclose(corr.sum(axis=0), -e2, atol=1e-12)))
    z = L.bowditch(T[0], b, d)
    checks.append(("edge: a traverse with no misclosure is unchanged by Bowditch", np.allclose(z[0][:-1], T, atol=1e-9)))
    checks.append(("edge: relative accuracy of a perfect closure is infinite; 0.1347 m in 673.238 m is 1 in 5000",
                   np.isinf(L.relative_accuracy(0.0, P)) and abs(L.relative_accuracy(0.13465, P) - 5000.0) < 2.0))

    # Monte Carlo with typical instrument figures
    mc = L.monte_carlo()
    rms_raw = float(np.sqrt(np.mean(mc["raw_unadjusted"] ** 2)))
    pred = L.predicted_misclosure_rms()
    checks.append(("validation: Monte Carlo RMS misclosure 39.4 mm matches the first-order prediction 39.8 mm within 3 percent",
                   abs(rms_raw * 1000 - 39.4) < 0.5 and abs(pred * 1000 - 39.8) < 0.5 and abs(rms_raw / pred - 1.0) < 0.03))
    checks.append(("typical errors: the median relative accuracy is about 1 in 30000, better than 1 in 5000",
                   25000 < np.median(mc["relative"]) < 35000 and np.median(mc["relative"]) > 5000))
    checks.append(("typical errors: the area error has a standard deviation of 1.17 m2 and no bias beyond 0.05 m2",
                   abs(mc["area_err"].std() - 1.17) < 0.05 and abs(mc["area_err"].mean()) < 0.05))
    checks.append(("validation: the mean adjusted area is 30000 m2 within 0.05 m2",
                   abs(mc["area_err"].mean()) < 0.05))
    ang_only = L.monte_carlo(trials=3000, dist_scale=0.0)
    dis_only = L.monte_carlo(trials=3000, angle_arcsec=0.0)
    ra = float(np.sqrt(np.mean(ang_only["raw_unadjusted"] ** 2))) * 1000
    rd = float(np.sqrt(np.mean(dis_only["raw_unadjusted"] ** 2))) * 1000
    checks.append(("sensitivity: angles alone give 37 mm RMS misclosure, distances alone 13 mm; angles dominate",
                   abs(ra - 37.1) < 1.5 and abs(rd - 12.6) < 0.8 and ra > 2.5 * rd))
    checks.append(("sensitivity: distance errors dominate the area error (1.16 m2 against 0.18 m2 from angles alone)",
                   abs(dis_only["area_err"].std() - 1.16) < 0.08 and abs(ang_only["area_err"].std() - 0.18) < 0.03))
    r160 = np.sqrt(np.mean(L.monte_carlo(trials=1500, angle_arcsec=160.0)["raw_unadjusted"] ** 2)) * 1000
    r5 = np.sqrt(np.mean(L.monte_carlo(trials=1500, angle_arcsec=5.0)["raw_unadjusted"] ** 2)) * 1000
    checks.append(("sensitivity: 5 arc-seconds gives 16 mm and 160 arc-seconds gives 298 mm, above the 134.6 mm of 1 in 5000",
                   abs(r5 - 15.8) < 1.0 and abs(r160 - 297.6) < 12.0 and r160 > P / 5000.0 * 1000))

    r40 = np.sqrt(np.mean(L.monte_carlo(trials=1500, angle_arcsec=40.0)["raw_unadjusted"] ** 2)) * 1000
    r80 = np.sqrt(np.mean(L.monte_carlo(trials=1500, angle_arcsec=80.0)["raw_unadjusted"] ** 2)) * 1000
    lim = P / 5000.0
    checks.append(("sensitivity: 40 arc-seconds gives 75.6 mm (inside 134.6 mm), 80 arc-seconds gives 149.5 mm (outside)",
                   abs(r40 - 75.6) < 3.0 and abs(r80 - 149.5) < 5.0 and r40 < lim * 1000 < r80))
    x1, x2 = L.max_angle_sigma(lim), L.max_angle_sigma(lim / 2.0)
    checks.append(("design: RMS misclosure equal to the 1 in 5000 limit allows 71 arc-seconds; half the limit allows 35 arc-seconds",
                   abs(x1 - 71.1) < 0.2 and abs(x2 - 35.1) < 0.2))
    ok35 = np.mean(L.monte_carlo(trials=2000, angle_arcsec=35.0, seed=5)["raw_unadjusted"] <= lim)
    ok40 = np.mean(L.monte_carlo(trials=2000, angle_arcsec=40.0, seed=5)["raw_unadjusted"] <= lim)
    checks.append(("design: with 35 arc-seconds about 97 percent of surveys meet 1 in 5000, with 40 about 94 percent",
                   abs(ok35 - 0.97) < 0.015 and abs(ok40 - 0.94) < 0.015))
    checks.append(("worked example: misclosure (0.100, -0.060) m is 0.1166 m long, 1 in 5773 in 673.238 m",
                   abs(np.hypot(0.1, 0.06) - 0.1166) < 5e-5 and abs(P / np.hypot(0.1, 0.06) - 5773.0) < 1.0))
    ok80 = np.mean(L.monte_carlo(trials=2000, angle_arcsec=80.0, seed=5)["raw_unadjusted"] <= lim)
    checks.append(("design: with 80 arc-seconds only about 58 percent of surveys meet 1 in 5000", abs(ok80 - 0.58) < 0.03))
    checks.append(("hand calculation: 116.619 / 673.238 = 0.173 of the misclosure goes to a 116.619 m leg", abs(116.619 / P - 0.173) < 5e-4))
    checks.append(("hand calculation: 0.050 m in 673.238 m is 1 in 13465; a 116.619 m leg gets 0.0087 m",
                   abs(P / 0.05 - 13465.0) < 1.0 and abs(0.05 * 116.619 / P - 0.0087) < 5e-5))

    # curvature and refraction
    k = np.array([0.1, 0.5, 1.0, 2.0, 3.0, 5.0])
    ratio = L.cr_physics(k) / L.cr_formula(k)
    checks.append(("validation: 0.0675 K^2 matches the sphere geometry with refraction 0.14 within 0.02 percent for K = 0.1 to 5 km",
                   np.all(np.abs(ratio - 1.0) < 2e-4)))
    checks.append(("curvature alone at 1 km is 78.5 mm; with refraction the correction is 67.5 mm",
                   abs(float(L.sphere_drop(1000.0)) * 1000 - 78.48) < 0.05 and abs(float(L.cr_formula(1.0)) * 1000 - 67.5) < 1e-9))
    checks.append(("a 100 m sight needs only 0.675 mm", abs(float(L.cr_formula(0.1)) * 1000 - 0.675) < 1e-9))
    checks.append(("the exact sphere drop matches d^2/(2R) to 1e-9 relative at 3 km",
                   abs(float(L.sphere_drop(3000.0)) / (3000.0 ** 2 / (2 * L.R_EARTH)) - 1.0) < 2e-6))
    checks.append(("unequal sights of 30 m and 70 m leave 0.27 mm of net curvature and refraction error",
                   abs((float(L.cr_formula(0.07)) - float(L.cr_formula(0.03))) * 1000 - 0.27) < 1e-9))
    checks.append(("equal sights of 50 m and 50 m cancel exactly", abs(float(L.cr_formula(0.05) - L.cr_formula(0.05))) < 1e-15))

    # blunder
    ab = L.with_blunder(a, 3, 1.0)
    rb = L.adjust(T[0], b[0], ab, d)
    rawb = L.walk(T[0], L.bearings_from_angles(b[0], ab), d)
    checks.append(("failure case: one angle 1 degree wrong gives a 4.071 m raw misclosure and an angle sum 1 degree off",
                   abs(L.misclosure(rawb)[1] - 4.071) < 5e-4 and abs(L.angle_misclosure(ab) - 1.0) < 1e-9))
    checks.append(("failure case: after spreading the angle error the misclosure is 1.851 m, only 1 in 364",
                   abs(rb["misclosure"] - 1.851) < 5e-4 and abs(rb["relative"] - 364.0) < 1.0))
    checks.append(("failure case: Bowditch still closes it exactly (< 1e-9 m) so the figure looks perfect",
                   L.misclosure(rb["points"])[1] < 1e-9))
    err = np.hypot(*(rb["points"][:-1] - T).T)
    checks.append(("failure case: yet the corners are wrong by up to 0.56 m and the area by 10.5 m2",
                   abs(err.max() - 0.557) < 5e-3 and abs(rb["area"] - 30000.0 + 10.52) < 0.05))
    kk, size = L.locate_angle_blunder(T[0], b[0], ab, d)
    checks.append(("locator: names corner 4 (index 3) and estimates the blunder as 1.000 degree", kk == 3 and abs(size - 1.0) < 1e-6))
    found = 0
    rng = np.random.default_rng(7)
    for _ in range(300):
        c = int(rng.integers(1, 5))
        bb0, mm, mdd = L.simulate_field(T, rng)
        found += L.locate_angle_blunder(T[0], bb0, L.with_blunder(mm, c, 1.0), mdd)[0] == c
    checks.append(("locator: finds the right corner in all 300 seeded trials with typical noise and a 1 degree blunder", found == 300))
    dd = d.copy()
    dd[1] += 1.0
    mv = L.misclosure(L.walk(T[0], b, dd))[0]
    bm = float(np.degrees(np.arctan2(mv[0], mv[1])) % 360.0)
    checks.append(("a 1 m distance blunder on leg 2 gives a misclosure along leg 2 (bearing 59.04 degrees)",
                   abs(bm - b[1]) < 1e-6 and abs(np.hypot(*mv) - 1.0) < 1e-9))

    # mini-project
    fa, fd = np.array(L.FIELD_ANGLES), np.array(L.FIELD_DISTS)
    r0 = L.adjust(L.FIELD_START, L.FIELD_B0, fa, fd)
    checks.append(("mini-project: as given, the angles are 1.007 degrees off, the misclosure is 2.601 m, only 1 in 332",
                   abs(r0["angle_misclosure_deg"] - 1.0071) < 5e-4 and abs(r0["misclosure"] - 2.601) < 5e-4 and abs(r0["relative"] - 332.0) < 1.0))
    kf, sf = L.locate_angle_blunder(L.FIELD_START, L.FIELD_B0, fa, fd)
    checks.append(("mini-project: the blunder is at corner 4 and about 1.0 degree", kf == 3 and abs(sf - 1.0004) < 5e-4))
    fa2 = fa.copy()
    fa2[kf] -= round(sf, 1)
    r1 = L.adjust(L.FIELD_START, L.FIELD_B0, fa2, fd)
    checks.append(("mini-project: corrected, the angle misclosure is 26 arc-seconds and the closure is 1 in 20180",
                   abs(r1["angle_misclosure_deg"] * 3600 - 25.56) < 0.05 and abs(r1["relative"] - 20180.0) < 20.0))
    checks.append(("mini-project: the corrected area is 53300.35 m2 against the exact 53300 m2",
                   abs(r1["area"] - 53300.35) < 0.01 and abs(L.shoelace(L.FIELD_TRUTH) - 53300.0) < 1e-9))
    checks.append(("mini-project: every adjusted corner is within 0.04 m of the answer key",
                   np.hypot(*(r1["points"][:-1] - L.FIELD_TRUTH).T).max() < 0.04))
    checks.append(("mini-project: the corrected traverse closes exactly", L.misclosure(r1["points"])[1] < 1e-9))

    # the app model: fixed error patterns scaled by sliders
    pa = np.array([0.6, -1.0, 0.8, -0.3, 0.5])
    pd = np.array([1.0, -0.6, 0.8, -1.0, 0.5])

    def app(sig, ds, bl):
        am = a + sig * pa / 3600.0
        am[3] += bl
        return L.adjust(T[0], b[0], am, d + ds * pd / 1000.0)

    r320, r340 = app(320, 10, 0), app(340, 10, 0)
    checks.append(("app challenge: with 10 mm distance error, 320 arc-seconds still gives better than 1 in 5000, 340 does not",
                   r320["relative"] > 5000 and r340["relative"] < 5000))
    checks.append(("app model: 100 arc-seconds and 10 mm give a 0.037 m misclosure, 1 in 18053",
                   abs(app(100, 10, 0)["misclosure"] - 0.0373) < 5e-4 and abs(app(100, 10, 0)["relative"] - 18053) < 5))
    checks.append(("app model: the largest slider settings stay below a 2.2 m misclosure", app(600, 50, 1.0)["misclosure"] < 2.2))

    # roadmap numbers
    checks.append(("roadmap: the shipped Bowditch gives an RMS error of 16.3 mm at corner 5 in the 4000-trial Monte Carlo",
                   abs(np.sqrt(np.mean(mc["corner_err"] ** 2)) * 1000 - 16.3) < 0.1))
    checks.append(("roadmap: a 60 m x 40 m plot with 2 percent tilt, levelled to its mean height, has 360 m3 of cut",
                   abs(40.0 * 0.02 * 30.0 ** 2 / 2.0 - 360.0) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
